# NeuroScope AI - Notebook 00B: Dataset Verification

Run once after all datasets are in place.

- No Drive mounting - data lives locally
- No zip extraction - all datasets already extracted
- BASE and DS defined in every cell - run cells in any order

Cells:
1. Install packages (one-time, skip if venv ready)
2. Quick folder inspection
3. Fix misplaced files (template)
4. Full verification - all 16 datasets
5. Disk usage summary
6. File-type breakdown
7. Pipeline readiness summary

---

## Cell 1 - Install Packages (one-time only)

In [1]:
# One-time only. Skip if venv already has all packages.
import subprocess, sys

packages = [
    'monai[all]==1.4.0',
    'nibabel', 'SimpleITK', 'pydicom',
    'numpy<2',
    'albumentations', 'opencv-python',
    'efficientnet-pytorch', 'segmentation-models-pytorch',
    'grad-cam', 'onnx', 'onnxruntime-gpu',
    'lifelines', 'pycox', 'pyradiomics',
    'pyyaml', 'xlrd', 'openpyxl', 'kaggle',
    'pandas', 'matplotlib', 'seaborn', 'scikit-learn',
    'fastapi', 'uvicorn', 'anthropic',
]

print('Installing packages...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'  {status}  {pkg}')
print('Done. Restart kernel if this is the first install.')

Installing packages...
  FAILED  monai[all]==1.4.0
  OK  nibabel
  OK  SimpleITK
  OK  pydicom
  OK  numpy<2
  OK  albumentations
  OK  opencv-python
  OK  efficientnet-pytorch
  OK  segmentation-models-pytorch
  OK  grad-cam
  FAILED  onnx
  FAILED  onnxruntime-gpu
  OK  lifelines
  OK  pycox
  FAILED  pyradiomics
  OK  pyyaml
  OK  xlrd
  OK  openpyxl
  OK  kaggle
  OK  pandas
  OK  matplotlib
  OK  seaborn
  OK  scikit-learn
  OK  fastapi
  OK  uvicorn
  OK  anthropic
Done. Restart kernel if this is the first install.


---
## Cell 2 - Quick Folder Inspection

In [1]:
import os
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

for cancer in ['brain', 'lung', 'breast', 'liver', 'skin', 'spine']:
    cp = os.path.join(DS, cancer)
    if not os.path.exists(cp):
        print(f'\n{cancer}/  NOT FOUND')
        continue
    print(f'\n{cancer}/')
    for ds_name in sorted(os.listdir(cp)):
        dp = os.path.join(cp, ds_name)
        if not os.path.isdir(dp):
            continue
        try:
            items = os.listdir(dp)
            print(f'   {ds_name}/  [{len(items)} items]')
            for it in sorted(items)[:6]:
                fp = os.path.join(dp, it)
                if os.path.isdir(fp):
                    n = len(os.listdir(fp))
                    print(f'        {it}/  [{n} items]')
                else:
                    sz = os.path.getsize(fp) / (1024**2)
                    print(f'        {it}  ({sz:.1f} MB)')
            if len(items) > 6:
                print(f'        ... and {len(items)-6} more')
        except Exception as e:
            print(f'   error: {e}')


brain/
   brats2024/  [1 items]
        BraTS2024-BraTS-GLI-TrainingData/  [1 items]
   kaggle_brain_tumor/  [2 items]
        Testing/  [4 items]
        Training/  [4 items]
   utsw_glioma/  [3 items]
        UTSW_Glioma_2D/  [1 items]
        audit_summary.csv  (0.0 MB)
        dataset.csv  (5.0 MB)

lung/
   lidc_idri/  [5 items]
        LIDC-XML-only/  [2 items]
        lidc-idri-nodule-counts-6-23-2015.xlsx  (0.0 MB)
        lidc_idri/  [1010 items]
        metadata/  [1 items]
        tcia-diagnosis-data-2012-04-20.xls  (0.0 MB)
   luna16/  [12 items]
        annotations.csv  (0.1 MB)
        candidates_V2.csv  (68.7 MB)
        subset0/  [1 items]
        subset1/  [1 items]
        subset2/  [1 items]
        subset3/  [1 items]
        ... and 6 more
   nsclc_radiomics/  [3 items]
        NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv  (0.0 MB)
        metadata/  [1 items]
        nsclc_radiomics/  [422 items]

breast/
   cbis_ddsm/  [1 items]
        cbis_ddsm/  [2 it

---
## Cell 3 - Fix Misplaced Files

In [2]:
import os, shutil
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

# Uncomment any moves needed:
moves = [
    # (source, destination)
    # (os.path.join(DS, 'brain',  'Brain'),       os.path.join(DS, 'brain',  'kaggle_brain_tumor')),
    # (os.path.join(DS, 'skin',   'Skin cancer'), os.path.join(DS, 'skin',   'ham10000')),
    # (os.path.join(DS, 'skin',   'Melona'),      os.path.join(DS, 'skin',   'isic2020')),
    # (os.path.join(DS, 'breast', 'Breast'),      os.path.join(DS, 'breast', 'cbis_ddsm')),
]

if not moves:
    print('No moves configured - skip if data is already in correct folders.')
else:
    for src, dst in moves:
        if not os.path.exists(src):
            print(f'Source not found: {src}')
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src):
            s = os.path.join(src, item)
            d = os.path.join(dst, item)
            if os.path.exists(d):
                print(f'  Skip (exists): {item}')
                continue
            shutil.move(s, d)
            print(f'  Moved: {item}')
        try:
            os.rmdir(src)
            print(f'Done: {os.path.basename(src)} -> {os.path.basename(dst)}')
        except OSError:
            print(f'Source not empty after move: {src}')

No moves configured - skip if data is already in correct folders.


---
## Cell 4 - Full Dataset Verification

In [5]:
import os
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

def count_files(path, ext=None):
    total = 0
    for _, _, files in os.walk(path):
        for f in files:
            if ext is None:
                total += 1
            elif f.lower().endswith(tuple(ext) if isinstance(ext, (list, tuple)) else ext):
                total += 1
    return total

def folder_size_gb(path):
    total = 0
    for r, _, files in os.walk(path):
        for f in files:
            try: total += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return total / (1024**3)

DATASETS = [
    {'name': 'Brain   BraTS 2024',         'path': os.path.join(DS, 'brain',  'brats2024'),
     'min': 100, 'optional': False,
     'checks': [('NIfTI files present',  lambda p: count_files(p, ('.nii.gz', '.nii')) > 0),
                ('Patient folders exist', lambda p: any(os.path.isdir(os.path.join(p, d)) for d in os.listdir(p)))],
     'note': 'Patient folders each with t1n/t1c/t2w/t2f/seg .nii.gz'},

    {'name': 'Brain   Kaggle Brain Tumor', 'path': os.path.join(DS, 'brain',  'kaggle_brain_tumor'),
     'min': 5000, 'optional': False,
     'checks': [('Training/ folder', lambda p: os.path.isdir(os.path.join(p, 'Training'))),
                ('Testing/ folder',  lambda p: os.path.isdir(os.path.join(p, 'Testing'))),
                ('4 class subfolders', lambda p:
                    len([d for d in os.listdir(os.path.join(p,'Training'))
                         if os.path.isdir(os.path.join(p,'Training',d))]) >= 4
                    if os.path.isdir(os.path.join(p,'Training')) else False)],
     'note': 'Training/{glioma,meningioma,notumor,pituitary}/ + Testing/'},

    {'name': 'Brain   UTSW Glioma 2D',    'path': os.path.join(DS, 'brain',  'utsw_glioma'),
     'min': 50, 'optional': False,
     'checks': [('CSV label file',   lambda p: count_files(p, '.csv') > 0),
                ('Image files',      lambda p: count_files(p, ('.png','.jpg','.nii.gz')) > 0)],
     'note': 'images/ folder + dataset.csv with IDH/MGMT/1p19q labels'},

    {'name': 'Lung    LIDC-IDRI',          'path': os.path.join(DS, 'lung',   'lidc_idri'),
     'min': 100, 'optional': False,
     'checks': [('DICOM .dcm files',    lambda p: count_files(p, '.dcm') > 0),
                ('Metadata spreadsheet',lambda p: count_files(p, ('.xls','.xlsx','.csv')) > 0)],
     'note': 'LIDC-IDRI-* patient folders + metadata XLS'},

    {'name': 'Lung    LUNA16',             'path': os.path.join(DS, 'lung',   'luna16'),
     'min': 50, 'optional': False,
     'checks': [('MHD or DICOM files', lambda p: count_files(p, ('.mhd','.dcm','.raw')) > 0),
                ('CSV annotation',     lambda p: count_files(p, '.csv') > 0)],
     'note': 'subset0-9/ with .mhd/.raw + annotations.csv + candidates_V2.csv'},

    {'name': 'Lung    NSCLC-Radiomics',    'path': os.path.join(DS, 'lung',   'nsclc_radiomics'),
     'min': 10, 'optional': True,
     'checks': [('Files present', lambda p: count_files(p) > 0)],
     'note': 'LUNG1-* patient folders + clinical CSV (needed for NB18)'},

    {'name': 'Breast  CBIS-DDSM',          'path': os.path.join(DS, 'breast', 'cbis_ddsm'),
     'min': 1000, 'optional': False,
     'checks': [('Image files', lambda p: count_files(p, ('.jpg','.png','.dcm')) > 100),
                ('CSV metadata', lambda p: count_files(p, '.csv') > 0)],
     'note': 'csv/ + jpeg/ folders'},

    {'name': 'Breast  INbreast',            'path': os.path.join(DS, 'breast', 'inbreast'),
     'min': 100, 'optional': False,
     'checks': [('DICOM files',   lambda p: count_files(p, '.dcm') > 0),
                ('Metadata file', lambda p: count_files(p, ('.xls','.xlsx','.csv')) > 0)],
     'note': 'AllDICOMs/ + AllXML/ + INbreast.xls'},

    {'name': 'Breast  CMMD',                'path': os.path.join(DS, 'breast', 'cmmd'),
     'min': 100, 'optional': False,
     'checks': [('Patient folders or images', lambda p:
                    count_files(p, ('.dcm','.jpg','.png')) > 0 or
                    any(os.path.isdir(os.path.join(p,d)) for d in os.listdir(p))),
                ('Clinical data file', lambda p: count_files(p, ('.xlsx','.csv')) > 0)],
     'note': 'D1-* patient folders + CMMD_clinicaldata_revision.xlsx'},

    {'name': 'Breast  KAU-BCMD',            'path': os.path.join(DS, 'breast', 'kau_bcmd'),
     'min': 100, 'optional': False,
     'checks': [('Image files', lambda p: count_files(p, ('.jpg','.png','.dcm')) > 0)],
     'note': 'benign/ malignant/ normal/ class folders'},

    {'name': 'Liver   LiTS',                'path': os.path.join(DS, 'liver',  'lits'),
     'min': 10, 'optional': False,
     'checks': [('Volume/seg files', lambda p: count_files(p, ('.nii','.nii.gz','.png')) > 0),
                ('volume/seg naming', lambda p: any(
                    'volume' in f.lower() or 'segmentation' in f.lower()
                    for _,_,files in os.walk(p) for f in files))],
     'note': 'volume-*.nii + segmentations/ folder (or PNG version)'},

    {'name': 'Skin    HAM10000',             'path': os.path.join(DS, 'skin',   'ham10000'),
     'min': 5000, 'optional': False,
     'checks': [('HAM10000_metadata.csv', lambda p:
                    os.path.exists(os.path.join(p,'HAM10000_metadata.csv'))),
                ('Image part folders', lambda p:
                    any('part' in d.lower() or 'image' in d.lower()
                        for d in os.listdir(p)
                        if os.path.isdir(os.path.join(p,d))))],
     'note': 'HAM10000_images_part_1/ + part_2/ + HAM10000_metadata.csv'},

    {'name': 'Skin    ISIC 2020',            'path': os.path.join(DS, 'skin',   'isic2020'),
     'min': 1000, 'optional': False,
     'checks': [('train/ folder', lambda p: os.path.isdir(os.path.join(p,'train'))),
                ('CSV file',      lambda p: count_files(p, '.csv') > 0)],
     'note': 'train/ folder with images + train_concat.csv'},

    {'name': 'Skin    PH2',                  'path': os.path.join(DS, 'skin',   'ph2'),
     'min': 100, 'optional': False,
     'checks': [('Image files', lambda p: count_files(p, ('.bmp','.jpg','.png')) > 0)],
     'note': 'images/ + masks/ folders or dermoscopy images with metadata'},

    {'name': 'Spine   RSNA Lumbar',          'path': os.path.join(DS, 'spine',  'rsna_lumbar'),
     'min': 50, 'optional': False,
     'checks': [('train.csv',        lambda p: os.path.exists(os.path.join(p,'train.csv'))),
                ('train_images/ dir',lambda p: os.path.isdir(os.path.join(p,'train_images')))],
     'note': 'train_images/ + train.csv + train_label_coordinates.csv'},
]

print('=' * 72)
print('  NEUROSCOPE AI - FULL DATASET VERIFICATION')
print('=' * 72)

ready, missing, warn = [], [], []

for ds in DATASETS:
    path     = ds['path']
    optional = ds.get('optional', False)
    name     = ds['name']

    if not os.path.exists(path):
        print(f'  {"OPTIONAL" if optional else "MISSING":10}  {name}')
        if not optional: missing.append(name)
        continue

    n = count_files(path)
    if n == 0:
        print(f'  EMPTY       {name}')
        print(f'              {ds["note"]}')
        missing.append(name)
        continue

    check_results = []
    for cn, fn in ds.get('checks', []):
        try: ok = fn(path)
        except Exception: ok = False
        check_results.append((cn, ok))

    all_pass = all(r[1] for r in check_results)
    enough   = n >= ds['min']
    sz       = folder_size_gb(path)

    if all_pass and enough:
        print(f'  OK          {name:<35}  {n:>8,} files  {sz:>7.2f} GB')
        ready.append(name)
    else:
        print(f'  WARNING     {name:<35}  {n:>8,} files  {sz:>7.2f} GB')
        for cn, ok in check_results:
            print(f'    {"pass" if ok else "FAIL"}  {cn}')
        if not enough:
            print(f'    FAIL  Expected >= {ds["min"]:,} files, got {n:,}')
        print(f'    Note: {ds["note"]}')
        warn.append(name)
        ready.append(name)

print('=' * 72)
print(f'  Ready   : {len(ready)}')
if warn:    print(f'  Warnings: {len(warn)} (present but check structure)')
if missing: print(f'  Missing : {len(missing)} -> ' + ', '.join(missing))
print('=' * 72)

  NEUROSCOPE AI - FULL DATASET VERIFICATION
  OK          Brain   BraTS 2024                      6,750 files    35.35 GB
  OK          Brain   Kaggle Brain Tumor              7,200 files     0.16 GB
  OK          Brain   UTSW Glioma 2D                150,006 files     1.31 GB
  OK          Lung    LIDC-IDRI                     245,849 files   124.16 GB
  OK          Lung    LUNA16                          1,778 files   111.02 GB
  OK          Lung    NSCLC-Radiomics                52,075 files    33.32 GB
  OK          Breast  CBIS-DDSM                      10,243 files     5.87 GB
  OK          Breast  INbreast                        1,621 files     8.39 GB
  OK          Breast  CMMD                            5,204 files    21.29 GB
  OK          Breast  KAU-BCMD                        2,380 files     0.73 GB
  OK          Liver   LiTS                          175,918 files     4.19 GB
  OK          Skin    HAM10000                       10,020 files     2.70 GB
  OK          Skin  

---
## Cell 5 - Disk Usage Summary

In [5]:
import os
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

print('Disk Usage by Pipeline')
print('-' * 50)

total_f = 0
total_b = 0

for cancer in ['brain', 'lung', 'breast', 'liver', 'skin', 'spine']:
    cp = os.path.join(DS, cancer)
    if not os.path.exists(cp):
        continue
    n = sum(len(f) for _, _, f in os.walk(cp))
    sz = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(cp)
        for f in files
        if not os.path.islink(os.path.join(r, f))
    ) / (1024**3)
    total_f += n
    total_b += sz
    status = 'OK' if n > 0 else 'EMPTY'
    print(f'  {status:6}  {cancer:<8}  {n:>8,} files  {sz:>8.2f} GB')

print('-' * 50)
print(f'  Total     {total_f:>8,} files  {total_b:>8.2f} GB')

Disk Usage by Pipeline
--------------------------------------------------
  OK      brain      163,956 files     36.82 GB
  OK      lung       299,700 files    268.43 GB
  OK      breast      19,448 files     36.27 GB
  OK      liver      175,918 files      4.19 GB
  OK      skin        59,104 files      3.98 GB
  OK      spine      310,446 files     37.48 GB
--------------------------------------------------
  Total     1,028,572 files    387.18 GB


---
## Cell 6 - File-Type Breakdown Per Dataset

In [6]:
import os
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

def count_ext(path, exts):
    if not os.path.exists(path): return 0
    exts = tuple(exts) if isinstance(exts, list) else exts
    return sum(1 for _,_,files in os.walk(path) for f in files if f.lower().endswith(exts))

checks = [
    ('BraTS 2024',        os.path.join(DS,'brain','brats2024'),
     [('NIfTI .nii.gz',['.nii.gz']),('NIfTI .nii',['.nii'])]),
    ('Kaggle Brain',      os.path.join(DS,'brain','kaggle_brain_tumor'),
     [('JPEG',['.jpg','.jpeg']),('PNG',['.png'])]),
    ('UTSW Glioma 2D',   os.path.join(DS,'brain','utsw_glioma'),
     [('PNG/JPG',['.png','.jpg']),('CSV',['.csv'])]),
    ('LIDC-IDRI',         os.path.join(DS,'lung','lidc_idri'),
     [('DICOM',['.dcm']),('XML',['.xml']),('XLS',['.xls','.xlsx'])]),
    ('LUNA16',             os.path.join(DS,'lung','luna16'),
     [('MHD',['.mhd']),('RAW',['.raw']),('CSV',['.csv'])]),
    ('CBIS-DDSM',          os.path.join(DS,'breast','cbis_ddsm'),
     [('JPEG',['.jpg','.jpeg']),('CSV',['.csv'])]),
    ('INbreast',           os.path.join(DS,'breast','inbreast'),
     [('DICOM',['.dcm']),('XML',['.xml']),('XLS',['.xls','.xlsx'])]),
    ('CMMD',               os.path.join(DS,'breast','cmmd'),
     [('DICOM',['.dcm']),('XLSX',['.xlsx']),('CSV',['.csv'])]),
    ('KAU-BCMD',           os.path.join(DS,'breast','kau_bcmd'),
     [('Images',['.jpg','.png','.dcm'])]),
    ('LiTS',               os.path.join(DS,'liver','lits'),
     [('NIfTI .nii',['.nii']),('NIfTI .nii.gz',['.nii.gz']),('PNG',['.png'])]),
    ('HAM10000',           os.path.join(DS,'skin','ham10000'),
     [('JPEG',['.jpg','.jpeg']),('CSV',['.csv'])]),
    ('ISIC 2020',          os.path.join(DS,'skin','isic2020'),
     [('JPEG',['.jpg','.jpeg']),('CSV',['.csv'])]),
    ('PH2',                os.path.join(DS,'skin','ph2'),
     [('BMP',['.bmp']),('PNG',['.png']),('TXT',['.txt'])]),
    ('RSNA Lumbar Spine',  os.path.join(DS,'spine','rsna_lumbar'),
     [('DICOM',['.dcm']),('CSV',['.csv'])]),
]

for name, path, types in checks:
    if not os.path.exists(path): continue
    total = sum(len(f) for _,_,f in os.walk(path))
    if total == 0: continue
    print(f'\n{name}  (path: ...{path[-40:]})')
    for label, exts in types:
        n = count_ext(path, exts)
        if n > 0:
            print(f'  {label:<22} {n:>8,}')


BraTS 2024  (path: ...e\NeuroScope_AI\datasets\brain\brats2024)
  NIfTI .nii.gz             6,750

Kaggle Brain  (path: ...ope_AI\datasets\brain\kaggle_brain_tumor)
  JPEG                      7,200

UTSW Glioma 2D  (path: ...NeuroScope_AI\datasets\brain\utsw_glioma)
  PNG/JPG                 150,000
  CSV                           4

LIDC-IDRI  (path: ...ve\NeuroScope_AI\datasets\lung\lidc_idri)
  DICOM                   244,527
  XML                       1,319
  XLS                           2

LUNA16  (path: ...drive\NeuroScope_AI\datasets\lung\luna16)
  MHD                         888
  RAW                         888

CBIS-DDSM  (path: ...\NeuroScope_AI\datasets\breast\cbis_ddsm)
  JPEG                     10,237
  CSV                           6

INbreast  (path: ...e\NeuroScope_AI\datasets\breast\inbreast)
  DICOM                       410
  XML                         544
  XLS                           1

CMMD  (path: ...drive\NeuroScope_AI\datasets\breast\cmmd)
  DICOM     

---
## Cell 7 - Pipeline Readiness Summary

In [7]:
import os
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

def has_data(path):
    if not os.path.exists(path): return False
    return any(True for _,_,f in os.walk(path) if f)

def file_count(path):
    return sum(len(f) for _,_,f in os.walk(path))

pipelines = [
    ('Brain  (NB 03-04)', [
        ('BraTS 2024',       os.path.join(DS,'brain','brats2024')),
        ('Kaggle Brain MRI', os.path.join(DS,'brain','kaggle_brain_tumor')),
        ('UTSW Glioma 2D',  os.path.join(DS,'brain','utsw_glioma')),
    ]),
    ('Lung   (NB 05)', [
        ('LIDC-IDRI',        os.path.join(DS,'lung','lidc_idri')),
        ('LUNA16',           os.path.join(DS,'lung','luna16')),
        ('NSCLC-Radiomics',  os.path.join(DS,'lung','nsclc_radiomics')),
    ]),
    ('Breast (NB 07)', [
        ('CBIS-DDSM',        os.path.join(DS,'breast','cbis_ddsm')),
        ('INbreast',         os.path.join(DS,'breast','inbreast')),
        ('CMMD',             os.path.join(DS,'breast','cmmd')),
        ('KAU-BCMD',         os.path.join(DS,'breast','kau_bcmd')),
    ]),
    ('Liver  (NB 06)', [
        ('LiTS',             os.path.join(DS,'liver','lits')),
    ]),
    ('Skin   (NB 08)', [
        ('HAM10000',         os.path.join(DS,'skin','ham10000')),
        ('ISIC 2020',        os.path.join(DS,'skin','isic2020')),
        ('PH2',              os.path.join(DS,'skin','ph2')),
    ]),
    ('Spine  (NB 08B)', [
        ('RSNA Lumbar',      os.path.join(DS,'spine','rsna_lumbar')),
    ]),
]

print('=' * 60)
print('  PIPELINE READINESS')
print('=' * 60)

all_ready = True
for pipeline, datasets in pipelines:
    pipeline_ok = all(has_data(p) for _, p in datasets)
    status = 'READY' if pipeline_ok else 'NOT READY'
    print(f'\n  {status:12}  {pipeline}')
    for ds_name, ds_path in datasets:
        if has_data(ds_path):
            n = file_count(ds_path)
            print(f'     OK      {ds_name:<25} {n:>8,} files')
        else:
            print(f'     MISSING {ds_name:<25} not found or empty')
            all_ready = False

print()
print('=' * 60)
if all_ready:
    print('  All pipelines ready - proceed to Notebook 01')
else:
    print('  Fix missing datasets then re-run this cell')
print('  Next: 01_Data_Exploration.ipynb')
print('=' * 60)

  PIPELINE READINESS

  READY         Brain  (NB 03-04)
     OK      BraTS 2024                   6,750 files
     OK      Kaggle Brain MRI             7,200 files
     OK      UTSW Glioma 2D             150,006 files

  READY         Lung   (NB 05)
     OK      LIDC-IDRI                  245,849 files
     OK      LUNA16                       1,776 files
     OK      NSCLC-Radiomics             52,075 files

  READY         Breast (NB 07)
     OK      CBIS-DDSM                   10,243 files
     OK      INbreast                     1,621 files
     OK      CMMD                         5,204 files
     OK      KAU-BCMD                     2,380 files

  READY         Liver  (NB 06)
     OK      LiTS                       175,918 files

  READY         Skin   (NB 08)
     OK      HAM10000                    48,631 files
     OK      ISIC 2020                   10,020 files
     OK      PH2                            453 files

  NOT READY     Spine  (NB 08B)
     MISSING RSNA Lumbar   

In [8]:
import os, shutil

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets', 'skin')

# Move contents to temp first
temp = os.path.join(DS, 'temp_swap')
os.makedirs(temp, exist_ok=True)

# Step 1: move ham10000 contents to temp
for item in os.listdir(os.path.join(DS, 'ham10000')):
    shutil.move(os.path.join(DS, 'ham10000', item), os.path.join(temp, 'ham_' + item))

# Step 2: move isic2020 contents to ham10000
for item in os.listdir(os.path.join(DS, 'isic2020')):
    shutil.move(os.path.join(DS, 'isic2020', item), os.path.join(DS, 'ham10000', item))

# Step 3: move temp (original ham) to isic2020
for item in os.listdir(temp):
    shutil.move(os.path.join(temp, item), os.path.join(DS, 'isic2020', item.replace('ham_', '')))

shutil.rmtree(temp)
print('Swapped ham10000 and isic2020')

Swapped ham10000 and isic2020


In [9]:
import os, shutil

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets', 'spine')

src = os.path.join(DS, 'rsna-2024-lsdd')
dst = os.path.join(DS, 'rsna_lumbar')

os.makedirs(dst, exist_ok=True)
for item in os.listdir(src):
    shutil.move(os.path.join(src, item), os.path.join(dst, item))
shutil.rmtree(src)
print('Moved rsna-2024-lsdd -> rsna_lumbar')

Moved rsna-2024-lsdd -> rsna_lumbar


In [4]:
import os, shutil

BASE  = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
outer = os.path.join(BASE, 'datasets', 'spine', 'rsna_lumbar')
inner = os.path.join(outer, 'rsna-2024-lumbar-spine-degenerative-classification')

for item in os.listdir(inner):
    src = os.path.join(inner, item)
    dst = os.path.join(outer, item)
    if not os.path.exists(dst):
        shutil.move(src, dst)
        print(f'Moved: {item}')

os.rmdir(inner)
print('Done')

Moved: models
Moved: processed
Moved: sample_submission.csv
Moved: test_images
Moved: test_series_descriptions.csv
Moved: train.csv
Moved: train_images
Moved: train_label_coordinates.csv
Moved: train_series_descriptions.csv
Done


In [6]:
import os
UTSW_PATH = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\datasets\brain\utsw_glioma'

# Just show 10 filenames
count = 0
for r, d, files in os.walk(UTSW_PATH):
    for f in files:
        if f.lower().endswith(('.png', '.jpg')):
            print(f)
            count += 1
            if count >= 10:
                break
    if count >= 10:
        break

slice_053.png
slice_054.png
slice_055.png
slice_056.png
slice_057.png
slice_058.png
slice_059.png
slice_060.png
slice_061.png
slice_062.png


In [7]:
import os
UTSW_PATH = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\datasets\brain\utsw_glioma'

count = 0
for r, d, files in os.walk(UTSW_PATH):
    for f in files:
        if f.lower().endswith(('.png', '.jpg')):
            print(os.path.relpath(os.path.join(r, f), UTSW_PATH))
            count += 1
            if count >= 10:
                break
    if count >= 10:
        break

UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_053.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_054.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_055.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_056.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_057.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_058.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_059.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_060.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_061.png
UTSW_Glioma_2D\dataset\BT0001\brain_flair\slice_062.png


In [9]:
import pandas as pd, os

UTSW_PATH = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\datasets\brain\utsw_glioma'

# Find and peek at the CSV
for r, d, files in os.walk(UTSW_PATH):
    for f in files:
        if f == 'dataset.csv':
            df = pd.read_csv(os.path.join(r, f))
            print(df.head(3).to_string())
            print()
            print("dtypes:")
            print(df.dtypes)
            break

  PatientID  Slice                             Flair                             T1                             T1CE                             T2                                    Mask
0    BT0001     53  BT0001/brain_flair/slice_053.png  BT0001/brain_t1/slice_053.png  BT0001/brain_t1ce/slice_053.png  BT0001/brain_t2/slice_053.png  BT0001/segmentation_mask/slice_053.png
1    BT0001     54  BT0001/brain_flair/slice_054.png  BT0001/brain_t1/slice_054.png  BT0001/brain_t1ce/slice_054.png  BT0001/brain_t2/slice_054.png  BT0001/segmentation_mask/slice_054.png
2    BT0001     55  BT0001/brain_flair/slice_055.png  BT0001/brain_t1/slice_055.png  BT0001/brain_t1ce/slice_055.png  BT0001/brain_t2/slice_055.png  BT0001/segmentation_mask/slice_055.png

dtypes:
PatientID    object
Slice         int64
Flair        object
T1           object
T1CE         object
T2           object
Mask         object
dtype: object
  PatientID  Slice                             Flair                             T1  

In [10]:
import os, cv2, torch
import numpy as np, pandas as pd
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

BASE      = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS        = os.path.join(BASE, 'datasets')
UTSW_PATH = os.path.join(DS, 'brain', 'utsw_glioma')

def find_utsw_csv(root):
    for r, d, files in os.walk(root):
        for f in files:
            if f == 'dataset.csv':
                return os.path.join(r, f)
    return None

class UTSWGliomaDataset(Dataset):
    """
    UTSW Glioma 2D dataset.
    CSV columns: PatientID, Slice, Flair, T1, T1CE, T2, Mask
    Each row = one slice with 4 modality image paths + segmentation mask.
    Returns: stacked 4-channel image [4, H, W] + mask [H, W]
    """
    MODALITIES = ['Flair', 'T1', 'T1CE', 'T2']

    def __init__(self, root, transform=None, split='train', val_frac=0.15, seed=42):
        self.root      = root
        self.transform = transform
        self.samples   = []

        csv_path = find_utsw_csv(root)
        if csv_path is None:
            print('dataset.csv not found'); return

        df = pd.read_csv(csv_path)
        print(f'UTSW CSV: {len(df)} rows, {df["PatientID"].nunique()} patients')

        # Train/val split by patient (not by slice, to avoid leakage)
        patients = df['PatientID'].unique()
        tr_pids, va_pids = train_test_split(patients, test_size=val_frac, random_state=seed)
        split_pids = set(tr_pids if split == 'train' else va_pids)
        df_split = df[df['PatientID'].isin(split_pids)].reset_index(drop=True)

        for _, row in df_split.iterrows():
            mod_paths = {m: os.path.join(root, row[m].replace('/', os.sep)) 
                        for m in self.MODALITIES}
            mask_path = os.path.join(root, row['Mask'].replace('/', os.sep))
            self.samples.append((mod_paths, mask_path))

        print(f'UTSW [{split}]: {len(self.samples)} slices')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mod_paths, mask_path = self.samples[idx]

        # Load 4 modalities and stack -> [4, H, W]
        channels = []
        for m in self.MODALITIES:
            img = cv2.imread(mod_paths[m], cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((224, 224), dtype=np.uint8)
            channels.append(img)
        image = np.stack(channels, axis=-1)  # [H, W, 4]

        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros((224, 224), dtype=np.uint8)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask  = torch.from_numpy(mask).long()

        return image, mask

# ── Test ──────────────────────────────────────────────────────────────────
if os.path.exists(UTSW_PATH):
    tfm = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=[0.5]*4, std=[0.5]*4),  # 4-channel normalization
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})

    ds_utsw = UTSWGliomaDataset(UTSW_PATH, transform=tfm, split='train')
    if len(ds_utsw) > 0:
        img, mask = ds_utsw[0]
        print(f'Image shape : {img.shape}   (4 modalities)')
        print(f'Mask shape  : {mask.shape}')
        print(f'Mask unique : {torch.unique(mask).tolist()}')
        print('UTSWGliomaDataset OK')
else:
    print('UTSW path not found')

UTSW CSV: 30000 rows, 625 patients
UTSW [train]: 25488 slices
Image shape : torch.Size([4, 224, 224])   (4 modalities)
Mask shape  : torch.Size([224, 224])
Mask unique : [0]
UTSWGliomaDataset OK


In [13]:
import psutil
ram = psutil.virtual_memory()
print(f'Total RAM : {ram.total/1024**3:.1f} GB')
print(f'Available : {ram.available/1024**3:.1f} GB')
print(f'Used      : {ram.used/1024**3:.1f} GB')

Total RAM : 31.3 GB
Available : 2.0 GB
Used      : 29.2 GB
